In [0]:
# MAGIC %md # Modelos dbt — Gold refinado

# Modelo 1: Ranking de voltas por circuito
spark.sql("""
CREATE OR REPLACE TABLE f1_gold.fct_lap_rankings AS
SELECT
    r.circuit_short_name,
    r.country_name,
    r.name_acronym,
    r.full_name,
    r.team_name,
    r.fastest_lap_seconds,
    ROUND(r.fastest_lap_seconds / 60, 3) AS fastest_lap_minutes,
    RANK() OVER (
        PARTITION BY r.circuit_short_name
        ORDER BY r.fastest_lap_seconds ASC
    ) AS lap_rank
FROM f1_gold.fct_race_results r
WHERE r.fastest_lap_seconds IS NOT NULL AND r.fastest_lap_seconds > 0
""")
print("✅ fct_lap_rankings criado")

# Modelo 2: Campeonato com pontos reais da F1
spark.sql("""
CREATE OR REPLACE TABLE f1_gold.agg_championship_ranking AS
WITH points_per_race AS (
    SELECT
        driver_number, name_acronym, full_name, team_name,
        circuit_short_name, final_position,
        CASE final_position
            WHEN 1 THEN 25 WHEN 2 THEN 18 WHEN 3 THEN 15
            WHEN 4 THEN 12 WHEN 5 THEN 10 WHEN 6 THEN 8
            WHEN 7 THEN 6  WHEN 8 THEN 4  WHEN 9 THEN 2
            WHEN 10 THEN 1 ELSE 0
        END AS points
    FROM f1_gold.fct_race_results
    WHERE final_position IS NOT NULL
)
SELECT
    driver_number, name_acronym, full_name, team_name,
    COUNT(circuit_short_name) AS races_completed,
    SUM(points) AS total_points,
    SUM(CASE WHEN final_position = 1 THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN final_position <= 3 THEN 1 ELSE 0 END) AS podiums,
    ROUND(AVG(final_position), 2) AS avg_position,
    RANK() OVER (ORDER BY SUM(points) DESC) AS championship_rank
FROM points_per_race
GROUP BY driver_number, name_acronym, full_name, team_name
ORDER BY total_points DESC
""")
print("✅ agg_championship_ranking criado")

# Modelo 3: Pit stop por equipe
spark.sql("""
CREATE OR REPLACE TABLE f1_gold.fct_team_pit_performance AS
SELECT
    team_name,
    COUNT(*) AS total_pit_stops,
    ROUND(MIN(avg_pit_seconds), 3) AS fastest_avg_pit,
    ROUND(AVG(avg_pit_seconds), 3) AS season_avg_pit,
    ROUND(MIN(fastest_pit_seconds), 3) AS absolute_fastest_pit,
    RANK() OVER (ORDER BY AVG(avg_pit_seconds) ASC) AS pit_efficiency_rank
FROM f1_gold.fct_pit_stop_analysis
WHERE team_name IS NOT NULL
GROUP BY team_name
ORDER BY season_avg_pit ASC
""")
print("✅ fct_team_pit_performance criado")

# Mostra o campeonato!
spark.sql("SELECT * FROM f1_gold.agg_championship_ranking ORDER BY championship_rank").show(20, truncate=False)